# Поиск по корпусу (CQL) — Google Colab

Порядок: запустите ячейки сверху вниз (**Среда выполнения → Выполнить всё**), затем меняйте `QUERY` и перезапускайте только ячейку «Запрос».

Ничего локально ставить не нужно: код поиска записан в ноутбук, корпус скачивается с GitHub.

Синтаксис: `[lemma="..."]`, `[word="..."]`, `[pos="..."]`, `[tag="..."]`, `&` внутри скобок, токены подряд, квантификаторы `? * + {n,m}`. Значения — регулярки (fullmatch); `|` группируйте: `.*(?:чыко|чыку)`.

## 2. Корпус с GitHub

In [16]:
SCRIPT_URL = "https://github.com/katykool/chkchn_updated/blob/main/scripts/cql.py"
CORPUS_URL = "https://github.com/katykool/chkchn_updated/blob/main/corpora/charles-weinstein-morphology.jsonl"

from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
CASE_SENSITIVE = False

In [17]:
import re
import sys
import urllib.request
import pandas as pd


def to_raw_url(url):
    """github.com/user/repo/blob/branch/path  ->  raw.githubusercontent.com/user/repo/branch/path"""
    m = re.match(r"https://github\.com/([^/]+)/([^/]+)/blob/(.+)", url)
    if m:
        return "https://raw.githubusercontent.com/%s/%s/%s" % m.groups()
    return url


def download(url, dest, token=None):
    req = urllib.request.Request(to_raw_url(url))
    if token:
        req.add_header("Authorization", "token " + token)
    with urllib.request.urlopen(req) as resp, open(dest, "wb") as f:
        f.write(resp.read())
    return dest


download(SCRIPT_URL, "cql_search.py", GITHUB_TOKEN)   # имя файла в репозитории может быть любым
sys.modules.pop("cql_search", None)                   # чтобы при повторном запуске взялась свежая версия
import cql_search as cql

records = cql.load_corpus(download(CORPUS_URL, "corpus.jsonl", GITHUB_TOKEN))
print("Предложений в корпусе:", len(records))


Предложений в корпусе: 20220


In [18]:
def search(query, limit=None, case_sensitive=CASE_SENSITIVE):
    """Возвращает (DataFrame, hits). Таблица = поля корпуса + столбец match."""
    pattern = cql.compile_pattern(cql.parse_query(query), case_sensitive)
    hits = cql.collect_hits(records, pattern, case_sensitive)
    if limit is not None:
        hits = hits[:limit]
    header, rows = cql.build_table(hits)
    return pd.DataFrame(rows, columns=header), hits


def save(hits, path="results.xlsx"):
    """Сохранить таблицу (.xlsx или .csv) и скачать её на компьютер."""
    if path.lower().endswith(".xlsx"):
        cql.write_xlsx(hits, path)
    else:
        cql.write_csv(hits, path)
    print("Сохранено:", path)
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        pass   # не в Colab — файл просто лежит рядом


## 3. Запрос

In [19]:
QUERY = '[lemma="пыкирык|эймэвык|ӄытык|тыԓек|эквэтык|рэк|рэсӄивык|пэԓӄынтэтык|тыттэтык|рыԓык" & tag!="caus" | word="эймэквъи" & tag!="caus"]'

LIMIT = None      # например 100, чтобы ограничить число совпадений

df, hits = search(QUERY, LIMIT)
print("Совпадений:", len(df))

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
df.drop(columns=["morphology"], errors="ignore")   # morphology скрыта для читаемости


Совпадений: 1308


,ckt,ru,score,source,article,translation,sent_id,match
0,"Авынвагты ыʼттъыт, аʼачекӄагтэ чейвэ аӈӄайпы ныԓейвыӄинэт","Хотя были собаки, юноши ходили на морскую охоту пешком по льду",0.659617,Charles_Weinstein,,,,ныԓейвыӄинэт
1,"Эвынъысӄэкут, авынваӈӄаттэ ӈирэӄ ԓевмиԓеӈыт, ынӄэнат рэвытрэнӈыт","Яловые важенки, две молодые неразлучные самки, они покажутся",0.816334,Charles_Weinstein,,,,рэвытрэнӈыт
2,Турыпкитԓьэт агагԓыкэгты ныԓеӄинэт,Только что приехавшие (люди) шли не торопясь,0.755337,Charles_Weinstein,,,,ныԓеӄинэт
3,ԓюур пыкиргъи миӈкэкин итрэтӄэй агтымкыкыԓьэн,вдруг откуда-то пришёл небольшой отряд,0.640836,Charles_Weinstein,,,,пыкиргъи
4,Айватԓьан-пынԓыԓьын ымы ынкы гэпкитԓин,Южанин-посыльный тоже туда пришёл,0.524260,Charles_Weinstein,,,,гэпкитԓин
...,...,...,...,...,...,...,...,...
1303,Ытԓывъёӄая пыкиринэӈу тэпӄэк йымнэнат вэԓвыегыт.,"Как только внук пришёл на косу, он надел снегоступы.",0.536804,Charles_Weinstein,,,,пыкиринэӈу
1304,Вуԓӄытвик ытԓён ӄытгъи яйваԓявыԓьэты.,"Вечером он пошёл к тем, которые осиротели.",0.663949,Charles_Weinstein,,,,ӄытгъи
1305,Мэмыԓ вэтгычемгъогъэ чьумэткук Кэӈъиринэ рээн.,Мэмыл согласился играть в шахматы с Кэӈъири.,0.712176,Charles_Weinstein,,,,рээн
1306,"ӄиквын ӈээкык, иӈӄун нытигыԓьэтыркын гымык рээн.","скажите дочери, пусть катается на лыжах со мной.",0.758152,Charles_Weinstein,,,,рээн


## 4. Сохранение
Таблица как во входном корпусе + столбец `match`. В Colab файл скачается через браузер.

In [ ]:
save(hits, "results.xlsx")


## Примеры запросов
```python
'[lemma="каргок"]'                       # любая форма леммы
'[word="вагъэ"]'                         # точная словоформа
'[pos="v" & tag="hab"]'                 # глагол в хабитуалисе
'[lemma="рэк"][word=".*чык[оу]"]'      # два токена подряд
'[pos="adv"]* [pos="v"]'                # квантификатор
```